In [1]:
from llm_consistency.core.paths import ProjectPaths
from llm_consistency.datasets.registry import get_dataset_spec
import pandas as pd

dataset = "SimpleQA"
experiment_flag = "v00"
temperature = 0.0

paths = ProjectPaths()
rp = paths.run_paths(dataset, experiment_flag)
conf = rp.conf_suffix(temperature=temperature)

d = pd.read_csv(rp.answers_all_models_file("both", conf))


df = pd.read_csv(rp.grades_all_judges_file("both"))


# print(d.shape)
print(df.shape)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None,
    'display.expand_frame_repr', False
):
    # display(df1[(df1.idx == 1499) & (df1.model == "gpt-3.5-turbo")])
    # display(df1)
    pass


(384006, 12)


In [2]:
df.idx.nunique()

1129

In [3]:
import pandas as pd

def aggregate_judges_list_based(df):
    qid_cols = ["model", "idx", "paraphrased_question"]

    grouped = (
        df
        .groupby(qid_cols)
        .agg(
            original_question=("original_question", "first"),
            verdicts=("verdict", list),
            evaluators=("evaluator", list),
        )
        .reset_index()
    )

    # ---- overall stats ----
    def overall_stats(vs):
        vc = pd.Series(vs).value_counts()
        return pd.Series({
            "num_judges": len(vs),
            "num_unique": vc.size,
            "all_same": vc.size == 1,
            "max_agreement_count": vc.max(),
            "max_agreement_value": vc.idxmax(),
        })

    grouped[[
        "num_judges",
        "num_unique",
        "all_same",
        "max_agreement_count",
        "max_agreement_value",
    ]] = grouped["verdicts"].apply(overall_stats)

    # ---- GPT-only stats ----
    def gpt_stats(row):
        gpt_verdicts = [
            v for v, e in zip(row["verdicts"], row["evaluators"])
            if "gpt" in e
        ]

        if len(gpt_verdicts) == 0:
            return pd.Series({
                "gpt_num_judges": 0,
                "gpt_num_unique": pd.NA,
                "gpt_agree": pd.NA,
                "gpt_max_agreement_count": pd.NA,
                "gpt_max_agreement_value": pd.NA,
            })

        vc = pd.Series(gpt_verdicts).value_counts()
        return pd.Series({
            "gpt_num_judges": len(gpt_verdicts),
            "gpt_num_unique": vc.size,
            "gpt_agree": vc.size == 1,
            "gpt_max_agreement_count": vc.max(),
            "gpt_max_agreement_value": vc.idxmax(),
        })

    grouped[[
        "gpt_num_judges",
        "gpt_num_unique",
        "gpt_agree",
        "gpt_max_agreement_count",
        "gpt_max_agreement_value",
    ]] = grouped.apply(gpt_stats, axis=1)

    return grouped.drop(columns="verdicts")

agg = aggregate_judges_list_based(df)

print(agg.shape)


(64001, 15)


In [4]:


df = agg.copy()

vo = df[df.original_question == df.paraphrased_question].copy()   # from original_only group
vp = df[df.original_question != df.paraphrased_question].copy()  # from paraphrased_only group
print(vo.shape, vp.shape)
vo = vo[vo.gpt_agree & (vo.max_agreement_count >= 4)]
vp = vp[vp.gpt_agree & (vp.max_agreement_count >= 4)]
print(vo.shape, vp.shape)
# print(vo.columns)
# vo = vo[vo.gpt_agree & (vo.max_agreement_count >= 5)]
# vp = vp[vp.gpt_agree & (vp.max_agreement_count >= 5)]
# print(vo.shape, vp.shape)

(7903, 15) (56098, 15)
(6979, 15) (48562, 15)


## HERE the actual analysis begins!

In [5]:
def original_distribution(df_orig, col="max_agreement_value", suffix="_orig"):
    """
    df_orig: verdicts_original (one row per (idx, model))
    Returns: DataFrame indexed by (idx, model) with columns A_orig, B_orig, C_orig
    """
    dist = pd.get_dummies(df_orig[col]) #\
            #    .reindex(columns=["A", "B", "C"], fill_value=0)

    # dist.columns = [c + "_orig" for c in dist.columns]
    dist.columns = dist.columns.astype(str) + suffix

    # Set multi-index (idx, model) for alignment with paraphrased
    dist["idx"] = df_orig["idx"]
    dist["model"] = df_orig["model"]
    return dist.set_index(["idx", "model"])

def paraphrased_distribution(df_para, col="max_agreement_value", suffix="_para"):
    """
    df_para: verdicts_paraphrased (many rows per (idx, model))
    Returns: normalized distribution per (idx, model) with A_para, B_para, C_para
    """
    norm = (
        df_para.groupby(["idx", "model"])[col]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        # .reindex(columns=["A", "B", "C"], fill_value=0)
    )

    # normalize row-wise inside each (idx, model)
    # norm = counts.div(counts.sum(axis=1), axis=0)
    # norm.columns = [c + suffix for c in norm.columns]
    norm.columns = norm.columns.astype(str) + suffix

    return norm

import numpy as np
def iid_mismatch_metric(group, col="max_agreement_value", suffix=""):
    p = group[col].value_counts(normalize=True)
    iid_mismatch_prob = 1 - (p ** 2).sum()
    distinct_answers = p.size
    mode_share = p.max()
    entropy = -(p * np.log(p)).sum()

    res = {
        "iid_mismatch_prob": iid_mismatch_prob,
        "distinct_classes": int(distinct_answers), # this is the number of distinct answers or labels
        "mode_share": mode_share,
        "entropy": entropy,
        "num_paraphrases": len(group),
        "normalized_entropy": entropy / np.log(len(group)) if len(group) > 1 else 0.0,
    }
    if suffix:
        res = {k + suffix: v for k, v in res.items()}
    return pd.Series(res)


In [6]:
vpp = pd.concat([vp, vo], ignore_index=True)
orig_dist = original_distribution(vo)
para_dist = paraphrased_distribution(vp)
para_dist2 = vp.groupby(["idx", "model"])[["max_agreement_value"]].apply(iid_mismatch_metric)
# para_dist2 = vp.groupby(["idx", "model"])["max_agreement_value"].apply(iid_mismatch_metric_df)
para_dist2["distinct_classes"] = para_dist2["distinct_classes"].astype("int64")
para_dist2["num_paraphrases"] = para_dist2["num_paraphrases"].astype("int64")

su = "_both"
both_dist = paraphrased_distribution(vpp, suffix=su)
both_dist2 = vpp.groupby(["idx", "model"])[["max_agreement_value"]].apply(iid_mismatch_metric, suffix=su)
both_dist2["distinct_classes"+su] = both_dist2["distinct_classes"+su].astype("int64")
both_dist2["num_paraphrases"+su] = both_dist2["num_paraphrases"+su].astype("int64")


print(orig_dist.shape, para_dist.shape, para_dist2.shape, both_dist.shape, both_dist2.shape)

# merged_dist = orig_dist.join(para_dist, how="inner")
merged = (
    orig_dist
    .join(para_dist, how="inner")
    .join(para_dist2, how="inner")
)
merged2 = (
    orig_dist
    .join(para_dist, how="inner")
    .join(para_dist2, how="inner")
    .join(both_dist, how="inner")
    .join(both_dist2, how="inner")
)
print(merged.shape, merged2.shape)
aligned = merged.copy()
aligned = merged2.copy()
print(aligned.columns)
for subset in ["orig", "para", "both"]:
    labels = ["correct_", "incorrect_", "not_attempted_"]
    cols = [label + subset for label in labels]

    aligned[f"{subset}_label"] = merged2[cols].idxmax(axis=1).str.replace(f"_{subset}", "")
    vals = merged2[cols]

    row_max = vals.max(axis=1)
    num_max = vals.eq(row_max, axis=0).sum(axis=1)

    # aligned[f"{subset}_label"] = np.where(
    #     num_max == 1,
    #     vals.idxmax(axis=1).str.replace(f"_{subset}", ""),
    #     "tie"
    # )

    aligned[f"{subset}_is_tie"] = num_max > 1
    def tied_labels(row):
        m = row.max()
        return tuple(
            c.replace(f"_{subset}", "")
            for c in cols if row[c] == m
        )

    aligned[f"{subset}_tied_labels"] = merged2[cols].apply(tied_labels, axis=1)


# aligned["orig_label"] = merged[["A_orig", "B_orig", "C_orig"]].idxmax(axis=1).str.replace("_orig", "")
# aligned["para_label"] = merged[["A_para", "B_para", "C_para"]].idxmax(axis=1).str.replace("_para", "")
# aligned["both_label"] = merged2[["A_both", "B_both", "C_both"]].idxmax(axis=1).str.replace("_both", "")
aligned["match"] = aligned["orig_label"] == aligned["para_label"]
# aligned["match_"] = aligned["orig_label"] == aligned["both_label"]
aligned


(6979, 3) (7630, 3) (7630, 6) (7673, 3) (7673, 6)
(6936, 12) (6936, 21)
Index(['correct_orig', 'incorrect_orig', 'not_attempted_orig', 'correct_para',
       'incorrect_para', 'not_attempted_para', 'iid_mismatch_prob',
       'distinct_classes', 'mode_share', 'entropy', 'num_paraphrases',
       'normalized_entropy', 'correct_both', 'incorrect_both',
       'not_attempted_both', 'iid_mismatch_prob_both', 'distinct_classes_both',
       'mode_share_both', 'entropy_both', 'num_paraphrases_both',
       'normalized_entropy_both'],
      dtype='object')


,,correct_orig,incorrect_orig,not_attempted_orig,correct_para,incorrect_para,not_attempted_para,iid_mismatch_prob,distinct_classes,mode_share,entropy,...,orig_label,orig_is_tie,orig_tied_labels,para_label,para_is_tie,para_tied_labels,both_label,both_is_tie,both_tied_labels,match
idx,model,,,,,,,,,,,,,,,,,,,,,
1,Qwen/Qwen2.5-7B-Instruct,False,False,True,0.000000,0.000000,1.000000,0.000000,1,1.000000,-0.000000,...,not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",True
2,Qwen/Qwen2.5-7B-Instruct,False,True,False,0.500000,0.250000,0.250000,0.625000,3,0.500000,1.039721,...,incorrect,False,"(incorrect,)",correct,False,"(correct,)",correct,True,"(correct, incorrect)",False
4,Qwen/Qwen2.5-7B-Instruct,False,False,True,0.000000,0.000000,1.000000,0.000000,1,1.000000,-0.000000,...,not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",True
5,Qwen/Qwen2.5-7B-Instruct,False,False,True,0.000000,0.000000,1.000000,0.000000,1,1.000000,-0.000000,...,not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",True
7,Qwen/Qwen2.5-7B-Instruct,False,False,True,0.000000,0.000000,1.000000,0.000000,1,1.000000,-0.000000,...,not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1492,meta-llama/Meta-Llama-3-8B-Instruct,False,False,True,0.000000,0.000000,1.000000,0.000000,1,1.000000,-0.000000,...,not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",True
1495,meta-llama/Meta-Llama-3-8B-Instruct,False,False,True,0.000000,0.000000,1.000000,0.000000,1,1.000000,-0.000000,...,not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",not_attempted,False,"(not_attempted,)",True
1496,meta-llama/Meta-Llama-3-8B-Instruct,False,False,True,0.000000,0.800000,0.200000,0.320000,2,0.800000,0.500402,...,not_attempted,False,"(not_attempted,)",incorrect,False,"(incorrect,)",incorrect,False,"(incorrect,)",False


In [7]:
tie_stats = {}

for subset in ["orig", "para", "both"]:
    labels = ["correct_", "incorrect_", "not_attempted_"]
    cols = [label + subset for label in labels]

    row_max = aligned[cols].max(axis=1)
    num_max = aligned[cols].eq(row_max, axis=0).sum(axis=1)

    tie_stats[subset] = {
        "num_rows": len(aligned),
        "num_ties": (num_max > 1).sum(),
        "tie_rate": (num_max > 1).mean(),
    }

tie_stats


{'orig': {'num_rows': 6936,
  'num_ties': np.int64(0),
  'tie_rate': np.float64(0.0)},
 'para': {'num_rows': 6936,
  'num_ties': np.int64(232),
  'tie_rate': np.float64(0.03344867358708189)},
 'both': {'num_rows': 6936,
  'num_ties': np.int64(179),
  'tie_rate': np.float64(0.025807381776239906)}}

In [8]:
# dist_cols = ["A_orig","B_orig","C_orig","A_para","B_para","C_para"]
dist_cols = []
for subset in ["orig", "para"]:
    labels = ["correct_", "incorrect_", "not_attempted_"]
    dist_cols += [label + subset for label in labels]

dist = aligned.groupby("model")[dist_cols].mean()
diff_df = aligned.assign(
    diff_A = aligned["correct_para"] - aligned["correct_orig"].astype(float),
    diff_B = aligned["incorrect_para"] - aligned["incorrect_orig"].astype(float),
    diff_C = aligned["not_attempted_para"] - aligned["not_attempted_orig"].astype(float),
).groupby("model")[["diff_A", "diff_B", "diff_C"]].mean()
total_counts = aligned.groupby("model").size().rename("total_samples")
agg_dist = dist.join(diff_df).join(total_counts)
agg_dist.sort_values("correct_orig", ascending=False).round(2)

,correct_orig,incorrect_orig,not_attempted_orig,correct_para,incorrect_para,not_attempted_para,diff_A,diff_B,diff_C,total_samples
model,,,,,,,,,,
gpt-4.1,0.47,0.39,0.14,0.45,0.37,0.17,-0.02,-0.02,0.04,893
gpt-4o,0.23,0.19,0.58,0.19,0.14,0.67,-0.03,-0.06,0.09,1010
gpt-4.1-mini,0.21,0.74,0.06,0.21,0.72,0.07,0.00,-0.02,0.01,829
gpt-3.5-turbo,0.10,0.31,0.58,0.09,0.22,0.69,-0.02,-0.09,0.11,988
meta-llama/Meta-Llama-3-8B-Instruct,0.04,0.34,0.62,0.04,0.36,0.61,-0.00,0.02,-0.01,991
meta-llama/Llama-3.1-8B-Instruct,0.01,0.03,0.96,0.01,0.02,0.96,0.00,-0.00,0.00,1112
Qwen/Qwen2.5-7B-Instruct,0.01,0.01,0.98,0.01,0.02,0.98,0.00,0.00,-0.00,1113


In [9]:
orig_dist = (
    aligned.groupby("model")["orig_label"]
    .value_counts(normalize=True)
    .unstack().fillna(0)
    # .add_prefix("orig_")
)

# Paraphrased label frequencies
para_dist = (
    aligned.groupby("model")["para_label"]
    .value_counts(normalize=True)
    .unstack().fillna(0)
    # .add_prefix("para_")
)

# Differences
label_diff = (para_dist - orig_dist)
orig_dist = orig_dist.add_suffix("_orig")
para_dist = para_dist.add_suffix("_maj_vote")
label_diff = label_diff.add_suffix("_diff")

total_counts = aligned.groupby("model").size().rename("total_samples")

agg_maj = (
    # dist
    # .join(diff_df)
    # .join()
        orig_dist
    .join(para_dist)
    .join(label_diff)
    .join(total_counts)
)

agg_maj.sort_values("correct_orig", ascending=False).round(2)

,correct_orig,incorrect_orig,not_attempted_orig,correct_maj_vote,incorrect_maj_vote,not_attempted_maj_vote,correct_diff,incorrect_diff,not_attempted_diff,total_samples
model,,,,,,,,,,
gpt-4.1,0.47,0.39,0.14,0.48,0.37,0.15,0.00,-0.02,0.01,893
gpt-4o,0.23,0.19,0.58,0.20,0.13,0.67,-0.03,-0.06,0.09,1010
gpt-4.1-mini,0.21,0.74,0.06,0.23,0.72,0.05,0.02,-0.02,-0.00,829
gpt-3.5-turbo,0.10,0.31,0.58,0.10,0.21,0.69,-0.00,-0.10,0.10,988
meta-llama/Meta-Llama-3-8B-Instruct,0.04,0.34,0.62,0.04,0.36,0.59,0.00,0.03,-0.03,991
meta-llama/Llama-3.1-8B-Instruct,0.01,0.03,0.96,0.01,0.02,0.97,0.00,-0.00,0.00,1112
Qwen/Qwen2.5-7B-Instruct,0.01,0.01,0.98,0.00,0.01,0.99,-0.00,-0.00,0.00,1113


In [10]:
mismatch_stats = (
    aligned
    .assign(mismatch = ~aligned["match"])
    .groupby("model")
    .agg(
        total_samples = ("match", "size"),
        num_mismatches = ("mismatch", "sum"),
        mismatch_rate = ("mismatch", "mean"),
        iid_mismatch_prob = ("iid_mismatch_prob", "mean"),
        normalized_entropy = ("normalized_entropy", "mean"),
        mode_share = ("mode_share", "mean"),
    )
)
mismatch_stats.sort_values('mismatch_rate', ascending=False).round(2)

,total_samples,num_mismatches,mismatch_rate,iid_mismatch_prob,normalized_entropy,mode_share
model,,,,,,
gpt-3.5-turbo,988,206,0.21,0.18,0.16,0.86
gpt-4o,1010,188,0.19,0.16,0.14,0.88
meta-llama/Meta-Llama-3-8B-Instruct,991,154,0.16,0.15,0.14,0.88
gpt-4.1,893,137,0.15,0.15,0.13,0.89
gpt-4.1-mini,829,76,0.09,0.10,0.10,0.92
meta-llama/Llama-3.1-8B-Instruct,1112,25,0.02,0.02,0.02,0.98
Qwen/Qwen2.5-7B-Instruct,1113,17,0.02,0.02,0.02,0.99


In [11]:
import numpy as np
import pandas as pd

def flip_stats(group: pd.DataFrame) -> pd.Series:
    orig_A = group["orig_label"] == "A"
    para_A = group["para_label"] == "A"
    ND = group["mode_share"] < 1

    base_A = orig_A.sum()
    base_notA = (~orig_A).sum()

    def safe_ratio(num, den):
        return np.nan if den == 0 else num / den

    return pd.Series({
        "A_to_notA":   safe_ratio((orig_A & (~para_A)).sum(), base_A),
        "notA_to_A":   safe_ratio((~orig_A & para_A).sum(), base_notA),
        "A_to_ND":     safe_ratio((orig_A & ND).sum(), base_A),
        "notA_to_ND":  safe_ratio((~orig_A & ND).sum(), base_notA),
        "ND":          ND.mean(),
    })

# 1. Base table
df_flip = aligned.groupby("model").apply(flip_stats).reset_index()

# 2. Convert flip rates to percentages
pct_cols = ["A_to_notA", "notA_to_A", "A_to_ND", "notA_to_ND", "ND"]
df_flip[pct_cols] = df_flip[pct_cols] * 100

# 3. Compute Orig Acc = P(A_orig)
orig_acc = (
    aligned.assign(correct_orig = aligned["orig_label"] == "A")
           .groupby("model")["correct_orig"]
           .mean() * 100
)

df_flip["Orig_Acc"] = df_flip["model"].map(orig_acc)

# 4. Reorder columns: Orig Acc first
df_flip = df_flip[
    ["model", "Orig_Acc"] + [c for c in df_flip.columns if c not in ["model", "Orig_Acc"]]
]
df_flip = df_flip.sort_values("Orig_Acc", ascending=False)
df_flip.round(2)

,model,Orig_Acc,A_to_notA,notA_to_A,A_to_ND,notA_to_ND,ND
0,Qwen/Qwen2.5-7B-Instruct,0.0,NaN,0.0,NaN,5.57,5.57
1,gpt-3.5-turbo,0.0,NaN,0.0,NaN,44.64,44.64
2,gpt-4.1,0.0,NaN,0.0,NaN,37.96,37.96
3,gpt-4.1-mini,0.0,NaN,0.0,NaN,27.62,27.62
4,gpt-4o,0.0,NaN,0.0,NaN,42.48,42.48
5,meta-llama/Llama-3.1-8B-Instruct,0.0,NaN,0.0,NaN,5.85,5.85
6,meta-llama/Meta-Llama-3-8B-Instruct,0.0,NaN,0.0,NaN,40.97,40.97


In [12]:
import numpy as np
import pandas as pd

def compute_A_any_stats(group: pd.DataFrame) -> pd.Series:
    """
    Compute A_any statistics per model, fully consistent with probability definitions.
    """

    # Basic event indicators
    orig_A = group["orig_label"] == "A"
    para_A = group["para_label"] == "A"        # majority-vote paraphrase label
    A_any  = group["correct_para"] > 0               # at least one paraphrase was correct
    # todo: hardcoded above

    # Denominators
    base_all     = len(group)
    base_origNot = (~orig_A).sum()
    base_paraNot = (~para_A).sum()

    def P(event, base):
        return np.nan if base == 0 else (event.sum() / base) * 100

    out = {
        # P(A_orig)
        "Orig_Acc": P(orig_A, base_all),

        # P(A_any) — at least one paraphrase correct
        "A_any_all": P(A_any, base_all),

        # P(A_any | orig ≠ A)
        "A_any_given_origWrong": P(A_any & (~orig_A), base_origNot),

        # P(A_any | para majority-vote ≠ A)
        "A_any_given_paraWrong": P(A_any & (~para_A), base_paraNot),
    }

    return pd.Series(out)


# 1. Compute for all models
df_Aany = (
    aligned.groupby("model")
           .apply(compute_A_any_stats)
           .reset_index()
)

# 2. Sort by original accuracy
df_Aany = df_Aany.sort_values("Orig_Acc", ascending=False)
df_Aany.round(2)

,model,Orig_Acc,A_any_all,A_any_given_origWrong,A_any_given_paraWrong
0,Qwen/Qwen2.5-7B-Instruct,0.0,1.44,1.44,1.44
1,gpt-3.5-turbo,0.0,16.30,16.30,16.30
2,gpt-4.1,0.0,57.11,57.11,57.11
3,gpt-4.1-mini,0.0,29.43,29.43,29.43
4,gpt-4o,0.0,33.07,33.07,33.07
5,meta-llama/Llama-3.1-8B-Instruct,0.0,2.79,2.79,2.79
6,meta-llama/Meta-Llama-3-8B-Instruct,0.0,7.87,7.87,7.87


# END